<div style="font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif;max-width:900px;margin:0 auto;border-radius:16px;overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,0.12);border:1px solid #e2e2e2;">

<div style="background:linear-gradient(135deg,#002855 0%,#004b8d 55%,#0077c8 100%);padding:28px 20px 22px 20px;text-align:center;">
<img src="./img/ITESOLogo.png" alt="ITESO" width="260" style="margin-bottom:10px;">
<div style="color:#ffffff;font-size:15px;font-weight:600;letter-spacing:0.5px;text-transform:uppercase;opacity:0.9;">
Departamento de Electrónica, Sistemas e Informática
</div>
</div>

<div style="background-color:#ffffff;padding:26px 30px 30px 30px;text-align:center;">

<div style="color:#002855;font-size:26px;font-weight:800;margin-bottom:6px;">
Big Data Analysis
</div>

<div style="display:inline-block;background-color:#eaf4fb;color:#0077c8;font-size:13px;font-weight:700;padding:4px 14px;border-radius:20px;letter-spacing:0.5px;margin-bottom:22px;">
Autumn 2026
</div>


<hr style="border:none;border-top:2px solid #f0f0f0;margin:0 0 22px 0;">

<div style="background-color:#f7fafd;border-left:5px solid #0077c8;border-radius:8px;padding:14px 18px;text-align:left;margin-bottom:18px;">
<div style="font-size:12px;color:#7a7a7a;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">
Session 08
</div>
<div style="font-size:19px;color:#002855;font-weight:700;">
Spark SQL
</div>
</div>

<div style="font-size:14px;color:#444;margin-top:20px;">
<span style="font-weight:700;color:#002855;">Profesor:</span> Pablo Camarillo Ramírez
</div>

</div>
</div>

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
            .appName("SparkSQL") \
            .master("local[*]") \
            .config("spark.ui.port", "4040") \
            .getOrCreate()

sc = spark.sparkContext

In [ ]:
data = [
    (1, "Alice", 29),
    (2, "Bob", 35),
    (3, "Charlie", 41)
]

df = spark.createDataFrame(data, ['id', 'Name', 'Age'])
df.printSchema()

# Smart Factory example

In [ ]:
from datetime import datetime

factory_data = [
    ("M001", None, 75.3),
    ("M002", datetime(2026, 9, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 9, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 9, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 9, 26, 8, 20, 0), None),
    ("M001", datetime(2026, 9, 26, 8, 25, 0), 77.5),
    ("M003", None, 73.2),
    ("M002", datetime(2026, 9, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 9, 26, 8, 40, 0), None),
    ("M003", datetime(2026, 9, 26, 8, 45, 0), 74.6),
]

factory_df = spark.createDataFrame(factory_data, ['id', 'date', 'temp'])
factory_df.show()

## Transformations and Actions

In [ ]:
from pyspark.sql.functions import col
df_f = factory_df.filter(col("temp") > 100)
df_f.show()

In [ ]:
print(f"número de elementos en el DataFrame original: {factory_df.count()}")
print(f"número de elementos en el DataFrame filtered: {df_f.count()}")

In [ ]:
factory_df = factory_df.orderBy(col("temp"), ascending=False)
factory_df.show()

In [ ]:
factory_groupped = factory_df.groupBy(col("id")).count()
factory_groupped.show()

In [ ]:
from pyspark.sql.functions import avg, min
agg_df = factory_df.groupBy(col("id")).agg(
    avg("temp").alias("avg_temp"),
    min("temp").alias("min_temp")
)
agg_df.show()


## Data cleaning

In [ ]:
from pyspark.sql.functions import count, when, isnull

factory_df.select([count(when(isnull(c), c)).alias(c) for c in factory_df.columns]).show()

## Dropna

In [ ]:
clean_v1 = factory_df.dropna()
clean_v1.show()
clean_v1.select([count(when(isnull(c), c)).alias(c) for c in clean_v1.columns]).show()

### Fillna

In [ ]:
replace_dictionary = {
    'temp': 0.0,
    'date': datetime.now().strftime("%Y-%m-%d %H:%M:%S") # datetime format supported by Spark
}
clean_v2 = factory_df.fillna(replace_dictionary)
clean_v2.show()
clean_v2.select([count(when(isnull(c), c)).alias(c) for c in clean_v1.columns]).show()

# Lab3: Data cleaning and basic transformations

In [ ]:
ecommerce_columns = [
    "order_id", "customer_name", "city", "product_category", "quantity",
    "unit_price", "order_value", "customer_rating", "order_date",
    "session_id", "internal_flag",
]
 
ecommerce_data = [
    ('ORD001', 'Daniela Hernandez', 'Guadalajara', 'Sports', 2, 189.28, 378.56, None, '2026-02-13 17:50:00', 'S1001', 1),
    ('ORD002', 'Valentina Moreno', None, 'Books', 5, 86.15, 430.75, 3.9, '2026-02-07 15:55:00', 'S1002', 0),
    ('ORD003', 'Isabel Diaz', 'Puebla', 'Books', 3, 51.24, 153.72, 3.8, '2026-02-09 12:03:00', 'S1003', 0),
    ('ORD004', 'Andres Hernandez', 'Merida', 'Clothing', 5, 173.82, 869.1, None, '2026-02-09 13:30:00', 'S1004', 0),
    ('ORD005', 'Miguel Torres', 'Tijuana', 'Clothing', 4, 168.95, 675.8, 4.3, '2026-02-09 15:14:00', 'S1005', 0),
    ('ORD006', 'Javier Torres', 'Cancun', 'Home & Kitchen', 5, 464.56, 2322.8, 3.3, '2026-02-09 23:13:00', 'S1006', 0),
    ('ORD007', 'Isabel Ruiz', 'Tijuana', 'Electronics', 3, 721.51, 2164.53, 3.1, '2026-02-14 05:45:00', 'S1007', 0),
    ('ORD008', 'Marco Nunez', 'Puebla', 'Sports', 2, 198.65, 397.3, 4.1, '2026-02-07 07:29:00', 'S1008', 0),
    ('ORD009', 'Sofia Cruz', 'Leon', 'Clothing', 1, 355.55, 355.55, None, '2026-02-04 23:22:00', 'S1009', 0),
    ('ORD010', 'Gabriela Cruz', 'Queretaro', 'Electronics', 1, 1086.22, 1086.22, 4.1, '2026-02-07 09:43:00', 'S1010', 0),
    ('ORD011', 'Maria Diaz', 'Toluca', 'Home & Kitchen', 5, 556.31, 2781.55, 4.7, '2026-02-08 02:59:00', 'S1011', 0),
    ('ORD012', 'Jorge Jimenez', 'Toluca', 'Electronics', 5, 621.4, None, 3.0, '2026-02-09 14:14:00', 'S1012', 0),
    ('ORD013', 'Adriana Rojas', 'Monterrey', 'Clothing', 4, 341.13, 1364.52, 5.0, '2026-02-13 10:55:00', 'S1013', 0),
    ('ORD014', 'Laura Cruz', 'Cancun', 'Toys', 2, 282.23, 564.46, 4.5, '2026-02-05 21:51:00', 'S1014', 0),
    ('ORD015', 'Adriana Cruz', 'Leon', 'Clothing', 2, 151.9, 303.8, None, '2026-02-14 17:17:00', 'S1015', 0),
    ('ORD016', 'Carlos Sanchez', 'Monterrey', 'Electronics', 3, 292.11, 876.33, 3.5, '2026-02-12 09:06:00', 'S1016', 0),
    ('ORD017', 'Diego Sanchez', 'Leon', 'Toys', 2, 73.58, 147.16, 4.3, '2026-02-09 09:29:00', 'S1017', 0),
    ('ORD018', 'Daniela Hernandez', 'Guadalajara', 'Toys', 3, 250.15, 750.45, 3.2, '2026-02-05 16:38:00', 'S1018', 0),
    ('ORD019', 'Pedro Torres', 'Leon', 'Clothing', 1, 288.69, 288.69, 4.1, '2026-02-01 16:03:00', 'S1019', 0),
    ('ORD020', 'Diego Morales', 'Tijuana', 'Toys', 1, 91.16, 91.16, 3.0, '2026-02-10 05:13:00', 'S1020', 0),
    ('ORD021', 'Alejandro Jimenez', None, 'Home & Kitchen', 2, 248.35, None, 4.9, '2026-02-14 12:18:00', 'S1021', 0),
    ('ORD022', 'Diego Cruz', None, 'Home & Kitchen', 1, 580.39, 580.39, 3.2, '2026-02-05 13:29:00', 'S1022', 1),
    ('ORD023', 'Adriana Rojas', 'Tijuana', 'Electronics', 5, 306.58, 1532.9, 4.3, '2026-02-14 04:41:00', 'S1023', 0),
    ('ORD024', 'Pedro Diaz', 'Merida', 'Home & Kitchen', 3, 328.61, 985.83, 4.9, '2026-02-02 23:37:00', 'S1024', 1),
    ('ORD025', 'Andres Castro', 'Toluca', 'Sports', 2, 473.3, 946.6, 4.8, '2026-02-06 21:24:00', 'S1025', 0),
    ('ORD026', 'Sebastian Cruz', 'Guadalajara', 'Sports', 1, 475.57, 475.57, 3.3, '2026-02-03 23:02:00', 'S1026', 0),
    ('ORD027', 'Elena Vargas', 'Tijuana', 'Sports', 5, 295.41, None, 4.8, '2026-02-02 11:44:00', 'S1027', 1),
    ('ORD028', 'Monica Martinez', 'Queretaro', 'Home & Kitchen', 4, 375.84, 1503.36, 3.9, '2026-02-01 13:16:00', 'S1028', 0),
    ('ORD029', 'Carlos Gomez', 'Cancun', 'Home & Kitchen', 4, 163.72, 654.88, 3.6, '2026-02-02 05:47:00', 'S1029', 0),
    ('ORD030', 'Monica Jimenez', None, 'Home & Kitchen', 2, 532.3, 1064.6, 5.0, '2026-02-05 08:41:00', 'S1030', 0),
    ('ORD031', 'Marco Reyes', 'Tijuana', 'Sports', 2, 414.95, 829.9, 3.2, '2026-02-02 05:08:00', 'S1031', 0),
    ('ORD032', 'Miguel Sanchez', 'Tijuana', 'Electronics', 2, 718.01, 1436.02, 3.6, '2026-02-02 21:54:00', 'S1032', 0),
    ('ORD033', 'Veronica Jimenez', 'Mexico City', 'Electronics', 1, 1340.15, 1340.15, None, '2026-02-14 13:04:00', 'S1033', 0),
    ('ORD034', 'Alejandro Vargas', None, 'Clothing', 4, 367.84, 1471.36, 3.4, '2026-02-02 08:14:00', 'S1034', 0),
    ('ORD035', 'Alejandro Castro', 'Mexico City', 'Toys', 1, 287.22, 287.22, 4.8, '2026-02-08 11:26:00', 'S1035', 0),
    ('ORD036', 'Javier Reyes', 'Mexico City', 'Toys', 3, 188.6, 565.8, 3.4, '2026-02-09 23:03:00', 'S1036', 0),
    ('ORD037', 'Andres Lopez', 'Queretaro', 'Sports', 2, 271.96, 543.92, 4.2, '2026-02-08 15:59:00', 'S1037', 0),
    ('ORD038', 'Adriana Ramirez', 'Monterrey', 'Sports', 5, 365.54, 1827.7, 4.2, '2026-02-03 11:00:00', 'S1038', 0),
    ('ORD039', 'Elena Martinez', 'Guadalajara', 'Electronics', 2, 1479.12, 2958.24, 4.2, '2026-02-02 23:46:00', 'S1039', 0),
    ('ORD040', 'Patricia Morales', 'Merida', 'Books', 2, 112.17, 224.34, 3.0, '2026-02-03 18:12:00', 'S1040', 0),
    ('ORD041', 'Gabriela Ruiz', 'Toluca', 'Books', 1, 90.21, 90.21, 4.6, '2026-02-13 10:04:00', 'S1041', 0),
    ('ORD042', 'Sebastian Cruz', 'Merida', 'Groceries', 2, 94.37, 188.74, 3.9, '2026-02-07 05:33:00', 'S1042', 0),
    ('ORD043', 'Diego Sanchez', 'Queretaro', 'Groceries', 1, 91.35, 91.35, 3.5, '2026-02-08 23:25:00', 'S1043', 0),
    ('ORD044', 'Paola Martinez', 'Tijuana', 'Clothing', 4, 210.43, 841.72, 4.1, '2026-02-10 19:04:00', 'S1044', 1),
    ('ORD045', 'Paola Lopez', 'Cancun', 'Toys', 4, 51.47, 205.88, 3.7, '2026-02-10 04:59:00', 'S1045', 0),
    ('ORD046', 'Andres Moreno', 'Tijuana', 'Groceries', 2, 47.29, 94.58, 4.0, '2026-02-10 04:22:00', 'S1046', 0),
    ('ORD047', 'Gabriela Martinez', 'Cancun', 'Electronics', 4, 969.46, None, 4.3, '2026-02-03 05:50:00', 'S1047', 0),
    ('ORD048', 'Patricia Vargas', 'Tijuana', 'Groceries', 3, 53.75, None, 4.8, '2026-02-07 15:57:00', 'S1048', 0),
    ('ORD049', 'Maria Jimenez', 'Guadalajara', 'Beauty', 2, 173.03, None, 4.6, '2026-02-02 05:59:00', 'S1049', 0),
    ('ORD050', 'Sofia Sanchez', 'Puebla', 'Groceries', 1, 76.4, 76.4, 3.4, '2026-02-07 03:56:00', 'S1050', 0),
    ('ORD051', 'Luis Ramirez', 'Queretaro', 'Clothing', 5, 88.22, 441.1, None, '2026-02-09 20:58:00', 'S1051', 0),
    ('ORD052', 'Daniela Sanchez', 'Monterrey', 'Sports', 5, 422.21, 2111.05, 4.6, '2026-02-14 05:04:00', 'S1052', 0),
    ('ORD053', 'Valentina Vargas', 'Guadalajara', 'Toys', 4, 76.38, 305.52, 4.9, '2026-02-11 19:04:00', 'S1053', 0),
    ('ORD054', 'Laura Moreno', 'Toluca', 'Groceries', 4, 63.56, 254.24, 4.5, '2026-02-07 10:34:00', 'S1054', 0),
    ('ORD055', 'Gabriela Sanchez', 'Leon', 'Toys', 3, 57.17, 171.51, 4.7, '2026-02-05 11:18:00', 'S1055', 0),
    ('ORD056', 'Sebastian Diaz', 'Toluca', 'Electronics', 5, 1432.48, 7162.4, 3.2, '2026-02-10 13:57:00', 'S1056', 0),
    ('ORD057', 'Diego Nunez', 'Guadalajara', 'Clothing', 3, 150.91, 452.73, 4.4, '2026-02-08 07:13:00', 'S1057', 0),
]

eccomerce_df = spark.createDataFrame(ecommerce_data, ecommerce_columns)

# non-clean data:

eccomerce_df.select([count(when(isnull(c), c)).alias(c) for c in eccomerce_df.columns]).show()